# Bifrost NØX — WAF Web-Attack Classifier (Colab training)

Trains the 7-class web-attack classifier from the 4 downloaded datasets, then
exports **two** artifacts:

| artifact | what | drop-in? |
|---|---|---|
| `waf_baseline.joblib` | char-ngram TF-IDF + LogReg | **yes** — loads into current `predictor.py` unchanged |
| `waf_distilbert/` | fine-tuned DistilBERT (HF dir) | needs a predictor branch later (the "wire" step) |

**Taxonomy (7 classes, order is stable — do not reorder):**
`clean, sqli, xss, path_traversal, command_injection, ssti, scanner`

**Runtime:** set Colab to **GPU** (Runtime → Change runtime type → T4 GPU) for the DistilBERT cell.

**Datasets expected** (upload the 4 you downloaded, or put them in a Drive folder):
- `archive.zip` — payload dataset (`payload,length,attack_type,label`)
- `archive(1).zip` — CSIC 2010 (`csic_database.csv`, binary Normal/Anomalous → used for `clean`)
- `PayloadsAllTheThings-master.zip` — per-family payloads
- `SecLists-master.zip` — scanner/fuzz lists (**optional**; 712 MB — you can skip it, a scanner seed is built in)


In [ ]:
# 1. Install deps
!pip -q install "transformers>=4.46" "datasets>=2.20" accelerate evaluate "scikit-learn==1.8.0" joblib
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only")

In [ ]:
# 2. Get the data — choose ONE path.

# --- Path A: Google Drive (recommended for the big SecLists zip) ---
# Put the 4 zips in a Drive folder, set DATA_DIR to it.
USE_DRIVE = False
DATA_DIR = "/content/data"

import os
os.makedirs(DATA_DIR, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/bifrost_waf"   # <-- edit to your folder
else:
    # --- Path B: manual upload (skip SecLists if you don't want to wait) ---
    from google.colab import files
    print("Upload: archive.zip, archive(1).zip, PayloadsAllTheThings-master.zip, bifrost_clean_corpus.zip, bifrost_synth_forms.zip, bifrost_cmdi_seclists.zip, bifrost_synth_cmdi.zip, (optional) SecLists-master.zip")
    up = files.upload()
    for name in up:
        os.replace(name, os.path.join(DATA_DIR, name))

print("data dir:", DATA_DIR, "->", os.listdir(DATA_DIR))

In [ ]:
# 3. Unzip everything present
import zipfile, glob
EXTRACT = "/content/extracted"
os.makedirs(EXTRACT, exist_ok=True)

def unzip(name):
    p = os.path.join(DATA_DIR, name)
    if not os.path.exists(p):
        print("  (missing, skipping):", name); return None
    dest = os.path.join(EXTRACT, name.replace(".zip", ""))
    if not os.path.isdir(dest):
        with zipfile.ZipFile(p) as z:
            z.extractall(dest)
    print("  unzipped:", name, "->", dest)
    return dest

D_PAYLOAD = unzip("archive.zip")
D_CSIC    = unzip("archive(1).zip")
D_PATT    = unzip("PayloadsAllTheThings-master.zip")
D_SECL    = unzip("SecLists-master.zip")   # may be None
D_CLEAN   = unzip("bifrost_clean_corpus.zip")  # real benign traffic (FWAF/ECML/Zanbil/URL-rep), 2026-07-31
D_SYNTH   = unzip("bifrost_synth_forms.zip")   # synthetic login/checkout/search/contact/API bodies, 2026-07-31
D_CMDI    = unzip("bifrost_cmdi_seclists.zip") # SecLists commix command-injection payloads, 2026-07-31
D_SYNCMDI = unzip("bifrost_synth_cmdi.zip")    # raw operator+command cmdi (ml.waf.synth_cmdi), 2026-07-31

In [ ]:
# 4. Taxonomy + text normalization (mirrors ml/waf/taxonomy.py + sources.py)
import re

LABELS = ["clean","sqli","xss","path_traversal","command_injection","ssti","scanner"]
LABEL_TO_ID = {n:i for i,n in enumerate(LABELS)}
ATTACK = set(LABELS) - {"clean"}
MAX_LEN = 2048

def normalize_text(s):
    if not s: return ""
    s = s.replace("\x00","")
    s = re.sub(r"[ \t]+"," ", s.replace("\r\n","\n")).strip()
    return s[:MAX_LEN]

def rec(text, label, source):
    text = normalize_text(text)
    if not text or label not in LABEL_TO_ID: return None
    return {"text": text, "label": label, "source": source}

records = []
def add(text, label, source, bag=None):
    r = rec(text, label, source)
    if r: (bag if bag is not None else records).append(r)

In [ ]:
# 5a. Ingest the payload CSV (fsecurify-style). Real CSV parse (payloads contain commas).
import csv

PAYLOAD_MAP = {"norm":"clean","sqli":"sqli","xss":"xss",
               "path-traversal":"path_traversal","cmdi":"command_injection"}

def ingest_payload_csv():
    if not D_PAYLOAD: return
    path = os.path.join(D_PAYLOAD, "payload_full.csv")
    n = 0
    with open(path, encoding="utf-8", errors="ignore", newline="") as f:
        for row in csv.DictReader(f):
            raw = (row.get("attack_type") or "").strip()
            lab = PAYLOAD_MAP.get(raw)
            if lab:
                add(row.get("payload",""), lab, "payload_csv"); n += 1
    print("payload_csv ->", n)

ingest_payload_csv()

In [ ]:
# 5b. CSIC 2010 — binary only. Take Normal -> clean (URL text). Cap to keep balance.
CSIC_CLEAN_CAP = 8000

def ingest_csic():
    if not D_CSIC: return
    path = os.path.join(D_CSIC, "csic_database.csv")
    n = 0
    with open(path, encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f)
        label_col = reader.fieldnames[0]   # Normal/Anomalous sits in the unnamed 1st column
        for row in reader:
            cls = (row.get(label_col) or "").strip().lower()
            url = row.get("URL","") or ""
            content = row.get("content","") or ""
            text = (url + " " + content).strip()
            if cls == "normal":
                add(text, "clean", "csic"); n += 1
            if n >= CSIC_CLEAN_CAP: break
    print("csic clean ->", n)

ingest_csic()

In [ ]:
# 5c. PayloadsAllTheThings — directory name -> family. Extract payload-ish lines.
PATT_DIRS = {
    "sql injection":"sqli",
    "xss injection":"xss", "cross-site scripting":"xss",
    "directory traversal":"path_traversal", "file inclusion":"path_traversal",
    "command injection":"command_injection",
    "server side template injection":"ssti",
}
NOISE = re.compile(r"^(#{1,6}\s|>|\||!\[|\[.*\]\(|http[s]?://|```|---|\*\s|-\s\[)", re.I)

def looks_like_payload(line):
    line = line.strip()
    if len(line) < 4 or len(line) > MAX_LEN: return False
    if NOISE.match(line): return False
    # heuristic: has an attack-ish char pattern
    return bool(re.search(r"['\";<>{}|`$()]|\.\./|=|SELECT|script|UNION|passwd|\bcat\b", line, re.I))

def family_for(path):
    p = path.lower()
    for key, lab in PATT_DIRS.items():
        if key in p: return lab
    return None

def ingest_patt():
    if not D_PATT: return
    counts = {}
    for root, _, fnames in os.walk(D_PATT):
        lab = family_for(root)
        if not lab: continue
        for fn in fnames:
            if not fn.lower().endswith((".md",".txt")): continue
            fp = os.path.join(root, fn)
            try:
                with open(fp, encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        if looks_like_payload(line):
                            add(line, lab, "patt"); counts[lab]=counts.get(lab,0)+1
            except Exception: pass
    print("patt ->", counts)

ingest_patt()

In [ ]:
# 5d. SecLists (optional) -> scanner UAs + supplemental fuzz. Bounded.
SCANNER_CAP = 1500
def ingest_seclists():
    if not D_SECL:
        print("seclists absent -> using built-in scanner seed only"); return
    n = 0
    for root, _, fnames in os.walk(D_SECL):
        rl = root.lower()
        want_scanner = "user-agent" in rl or "user agents" in rl
        for fn in fnames:
            if not fn.lower().endswith(".txt"): continue
            if not want_scanner: continue
            fp = os.path.join(root, fn)
            try:
                with open(fp, encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        s = line.strip()
                        if re.search(r"sqlmap|nikto|nmap|nessus|dirbuster|acunetix|w3af|masscan|zgrab|wpscan|gobuster|ffuf|nuclei", s, re.I):
                            add(s, "scanner", "seclists"); n += 1
                        if n >= SCANNER_CAP: break
            except Exception: pass
            if n >= SCANNER_CAP: break
        if n >= SCANNER_CAP: break
    print("seclists scanner ->", n)

ingest_seclists()

In [ ]:
# 5e. Built-in seeds — guarantee ssti + scanner coverage even without SecLists.
SSTI_SEED = [
 "{{7*7}}", "${7*7}", "<%= 7*7 %>", "#{7*7}", "{{config.items()}}",
 "{{''.__class__.__mro__[1].__subclasses__()}}", "${T(java.lang.Runtime).getRuntime().exec('id')}",
 "{{request.application.__globals__.__builtins__.__import__('os').popen('id').read()}}",
 "*{7*7}", "@(7*7)", "#{ 7 * 7 }", "{{ ''.__class__ }}", "${{7*7}}", "{php}echo 7*7;{/php}",
]
SCANNER_SEED = [
 "sqlmap/1.7#stable (http://sqlmap.org)", "Mozilla/5.00 (Nikto/2.1.6)",
 "Nmap Scripting Engine", "Nessus", "dirbuster", "gobuster/3.6",
 "GET /?a=../../../../etc/passwd HTTP/1.1", "curl/7.88 (scan)", "Wfuzz/3.1",
 "Nuclei - Open-source project (github.com/projectdiscovery/nuclei)",
 "masscan/1.3", "acunetix-wvs", "w3af.org", "WPScan v3",
]
for s in SSTI_SEED: add(s, "ssti", "seed")
for s in SCANNER_SEED: add(s, "scanner", "seed")
print("seeds added. total records so far:", len(records))

In [ ]:
import json
# 5f. Bifrost clean corpus — real benign traffic (FWAF goodqueries, ECML/PKDD
# "Valid" requests, URL-reputation benign set, sampled Zanbil e-commerce nginx log).
# Fixes the FP root cause: CSIC-only clean data is lab URLs + news, never real
# webmail/API/form-POST traffic, so real benign requests were out-of-distribution.
def ingest_clean_corpus():
    if not D_CLEAN: return
    path = os.path.join(D_CLEAN, "bifrost_clean_corpus.jsonl")
    n = 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            r = json.loads(line)
            add(r["text"], "clean", r.get("source", "bifrost_clean_corpus")); n += 1
    print("bifrost_clean_corpus ->", n, "clean records")

ingest_clean_corpus()


In [ ]:
# 5g. Synthetic benign form/JSON bodies (Faker-generated, ml/waf/synth_benign.py).
# Closes the credential-POST-body gap: no public dataset carries real login/
# checkout/API bodies (privacy), so the retrained model still blocked
# "username=admin&password=..." at 0.999 confidence even after the clean-corpus
# retrain above. Structurally varied (5 form domains, multiple key-name variants)
# so the model learns the general shape, not one memorized pattern.
def ingest_synth_forms():
    if not D_SYNTH: return
    path = os.path.join(D_SYNTH, "synth_benign_forms.jsonl")
    n = 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            r = json.loads(line)
            add(r["text"], "clean", r.get("source", "synth_benign")); n += 1
    print("synth_benign_forms ->", n, "clean records")

ingest_synth_forms()


In [ ]:
# 5h. SecLists commix command-injection payloads (Fuzzing/command-injection-commix.txt).
# command_injection was the thinnest/weakest class (0.85/0.90 P/R on only 61 held-out
# samples) and regressed further after the clean+synth corpus grew — added volume here.
# NOTE: every line in this file is percent-encoded (%3B, %7C, ...); it does NOT cover
# raw/literal separators like the failing case "host=X; cat /etc/passwd". If that keeps
# failing after this retrain, the gap is encoding-diversity, not payload volume.
def ingest_cmdi_seclists():
    if not D_CMDI: return
    path = os.path.join(D_CMDI, "command-injection-commix.txt")
    n = 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            add(line, "command_injection", "seclists_commix"); n += 1
    print("seclists commix command_injection ->", n)

ingest_cmdi_seclists()


In [ ]:
# 5i. Synthetic RAW command-injection (ml/waf/synth_cmdi.py). Fixes the
# regression the commix corpus above did not: commix is 100% percent-encoded,
# so raw operators (';', '&&', '||', '|') combined with a form-key prefix
# (host=/cmd=/file=/ip=... — the SAME keys synth_benign.py uses for benign
# forms) were confidently predicted as clean (up to 0.994) instead of blocked.
def ingest_synth_cmdi():
    if not D_SYNCMDI: return
    path = os.path.join(D_SYNCMDI, "synth_cmdi_raw.jsonl")
    n = 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            r = json.loads(line)
            add(r["text"], "command_injection", r.get("source", "synth_cmdi_raw")); n += 1
    print("synth_cmdi_raw ->", n, "command_injection records")

ingest_synth_cmdi()


In [ ]:
# 6. Dedup + per-class cap for balance + distribution report
import collections, random
random.seed(42)

# exact-text dedup
seen, deduped = set(), []
for r in records:
    if r["text"] in seen: continue
    seen.add(r["text"]); deduped.append(r)

# cap majority classes so nothing drowns the model
CAPS = {"clean":20000, "sqli":6000}   # clean raised 2026-07-31: real-traffic corpus added (was CSIC-only)
by_lab = collections.defaultdict(list)
for r in deduped: by_lab[r["label"]].append(r)
balanced = []
for lab, items in by_lab.items():
    random.shuffle(items)
    cap = CAPS.get(lab)
    balanced += items[:cap] if cap else items

random.shuffle(balanced)
dist = collections.Counter(r["label"] for r in balanced)
print("FINAL distribution:")
for lab in LABELS: print(f"  {lab:20s} {dist.get(lab,0)}")
print("total:", len(balanced))
assert all(dist.get(l,0) > 0 for l in LABELS), "a class has 0 samples — check dataset paths"

In [ ]:
# 7. Stratified split 70/15/15
def stratified_split(recs, val=0.15, test=0.15, seed=42):
    rng = random.Random(seed)
    buckets = collections.defaultdict(list)
    for r in recs: buckets[r["label"]].append(r)
    tr, va, te = [], [], []
    for lab, items in buckets.items():
        rng.shuffle(items); n=len(items)
        nt=int(n*test); nv=int(n*val)
        te += items[:nt]; va += items[nt:nt+nv]; tr += items[nt+nv:]
    for s in (tr,va,te): rng.shuffle(s)
    return tr, va, te

train, val, test = stratified_split(balanced)
print("train/val/test:", len(train), len(val), len(test))
X_tr=[r["text"] for r in train]; y_tr=[r["label"] for r in train]
X_va=[r["text"] for r in val];   y_va=[r["label"] for r in val]
X_te=[r["text"] for r in test];  y_te=[r["label"] for r in test]

In [ ]:
# 8. BASELINE — char-ngram TF-IDF + LogReg (drops into current predictor.py)
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5),
                              min_df=1, lowercase=False, sublinear_tf=True)),
    ("clf", LogisticRegression(C=10.0, max_iter=2000, class_weight="balanced")),
])
pipe.fit(X_tr, y_tr)
present = sorted(set(y_te)|set(pipe.predict(X_te)), key=LABELS.index)
print(classification_report(y_te, pipe.predict(X_te), labels=present, zero_division=0))

# SAME bundle schema predictor.py expects: {"pipeline":..., "labels":...}
joblib.dump({"pipeline": pipe, "labels": LABELS}, "/content/waf_baseline.joblib")
print("saved /content/waf_baseline.joblib  <-- drop into ml/waf/artifacts/, no code change")

In [ ]:
# 9. DISTILBERT — fine-tune (needs GPU runtime)
import numpy as np
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import evaluate

MODEL = "distilbert-base-uncased"
id2label = {i:l for i,l in enumerate(LABELS)}
label2id = {l:i for i,l in enumerate(LABELS)}

tok = AutoTokenizer.from_pretrained(MODEL)
def to_ds(X, y):
    d = Dataset.from_dict({"text": X, "label": [label2id[l] for l in y]})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

ds_tr, ds_va, ds_te = to_ds(X_tr,y_tr), to_ds(X_va,y_va), to_ds(X_te,y_te)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

f1 = evaluate.load("f1"); acc = evaluate.load("accuracy")
def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": acc.compute(predictions=preds, references=p.label_ids)["accuracy"],
            "f1_macro": f1.compute(predictions=preds, references=p.label_ids, average="macro")["f1"]}

args = TrainingArguments(
    output_dir="/content/waf_bert_ckpt",
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=3e-5, num_train_epochs=3, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(), logging_steps=50, report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)
trainer.train()

In [ ]:
# 10. Evaluate DistilBERT on held-out test + confusion matrix vs baseline
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(ds_te)
y_pred = [LABELS[i] for i in np.argmax(pred.predictions, axis=1)]
print("=== DistilBERT test report ===")
present = sorted(set(y_te)|set(y_pred), key=LABELS.index)
print(classification_report(y_te, y_pred, labels=present, zero_division=0))
print("confusion (rows=true, cols=pred):", present)
print(confusion_matrix(y_te, y_pred, labels=present))

In [ ]:
# 11. Export both artifacts and download
import shutil, json
model.save_pretrained("/content/waf_distilbert")
tok.save_pretrained("/content/waf_distilbert")
with open("/content/waf_distilbert/labels.json","w") as f:
    json.dump(LABELS, f)
shutil.make_archive("/content/waf_distilbert", "zip", "/content/waf_distilbert")

from google.colab import files
files.download("/content/waf_baseline.joblib")   # -> ml/waf/artifacts/  (drop-in NOW)
files.download("/content/waf_distilbert.zip")     # -> for the wiring step later
print("done.")

## Next steps (after download)

1. **Baseline (drop-in now):** copy `waf_baseline.joblib` → `ml/waf/artifacts/waf_baseline.joblib`.
   `predictor.py` loads it unchanged (same `{"pipeline","labels"}` schema). Run the repo eval to confirm.
2. **DistilBERT (wiring step):** unzip `waf_distilbert.zip` into the repo. It does **not** fit the
   current joblib `predict_proba` contract — wiring it means adding a transformer branch to
   `predictor.py` (load HF model, softmax → same `WAFPrediction`). That's the "wire" task, separate
   from training.
3. Regression check before shipping either: benign financial-news text must **not** block.
